# Incertezas de los parámetros de ajuste ($\tau$, $\beta$)

Este notebook explica **qué hace `curve_fit`** y **de dónde salen las incertezas** (`tau_err`, `beta_err`) que se agregaron a `Polarización-vs-tiempo-combinado.py`.

El modelo ajustado es la exponencial estirada (Kohlrausch-Williams-Watts):

$$
P(t) = P_0 \, e^{-(t/\tau)^{\beta}} + P_\infty
$$

con $P_0 = 1$ y $P_\infty = 0$ fijos, y $\theta = (\tau, \beta)$ como parámetros libres.

## 1. Qué minimiza `curve_fit`

`curve_fit` busca los parámetros $\theta$ que minimizan la suma de residuos al cuadrado:

$$
\chi^2(\theta) = \sum_{i=1}^{N} \left[ y_i - f(x_i, \theta) \right]^2 = \sum_{i=1}^N r_i(\theta)^2
$$

donde $r_i(\theta) = y_i - f(x_i,\theta)$ es el residuo en el punto $i$, $N$ es el número de puntos de datos, y $f$ es `exponential_fit`.

Como $f$ **no es lineal** en $\theta$ (aparece $\tau$ y $\beta$ dentro de una potencia y una exponencial), no hay fórmula cerrada como en cuadrados mínimos lineales. SciPy resuelve esto de forma iterativa:

- Sin `bounds`: algoritmo de **Levenberg–Marquardt**.
- Con `bounds` (como en tu script): algoritmo **Trust Region Reflective (TRF)**.

Ambos parten de `p0` y en cada iteración linealizan el modelo alrededor del $\theta$ actual usando el **Jacobiano**:

$$
J_{ij} = \frac{\partial f(x_i, \theta)}{\partial \theta_j}
$$

(SciPy lo aproxima numéricamente por diferencias finitas, ya que no le pasaste derivadas analíticas). Con esa linealización resuelve un paso tipo cuadrados-mínimos-lineales, actualiza $\theta$, y repite hasta converger (gradiente de $\chi^2 \approx 0$).

## 2. De la curvatura de $\chi^2$ a la matriz de covarianza

La idea clave: **cerca del mínimo**, $\chi^2(\theta)$ se puede aproximar por una superficie cuadrática (expansión de Taylor a segundo orden alrededor de $\hat\theta$, el óptimo):

$$
\chi^2(\theta) \approx \chi^2(\hat\theta) + \tfrac{1}{2} (\theta-\hat\theta)^T H (\theta - \hat\theta)
$$

donde $H$ es la Hessiana de $\chi^2$. Para mínimos cuadrados, esa Hessiana se aproxima (ignorando términos de segundo orden del residuo, aproximación de Gauss-Newton) como:

$$
H \approx 2\, J^T J
$$

Si $\chi^2$ fuera exactamente cuadrático, esto define una **elipse (o elipsoide) de confianza** alrededor de $\hat\theta$: la curvatura en cada dirección te dice qué tan "ajustado" está ese parámetro por los datos.

- Curvatura alta (superficie muy "puntiaguda" en esa dirección) $\Rightarrow$ mover el parámetro empeora rápido el ajuste $\Rightarrow$ **incerteza chica**.
- Curvatura baja (superficie "plana") $\Rightarrow$ el parámetro está mal determinado $\Rightarrow$ **incerteza grande**.

La matriz de covarianza es, literalmente, la inversa de esa curvatura (salvo escala):

$$
\text{pcov} = s^2 \, (J^T J)^{-1}
$$

### ¿Qué es $s^2$?

Es la varianza residual estimada — cuánto "sobra" de dispersión entre el modelo y los datos que **no** se explica por la forma funcional:

$$
s^2 = \frac{\chi^2(\hat\theta)}{N - p}
$$

con $p$ = número de parámetros libres ($p=2$: $\tau,\beta$) y $N-p$ los **grados de libertad**.

> **Importante para tu caso:** como no le pasaste `sigma=` (incertezas experimentales conocidas de cada punto de $P/P_0$), `curve_fit` usa por defecto esta $s^2$ *inferida de los propios residuos*. Es decir, asume que todo el desvío entre el ajuste y los datos es "ruido", y reparte esa varianza entre los parámetros. Si tuvieras una incerteza conocida por punto, se pasaría con `sigma=` y `absolute_sigma=True`, y ahí `pcov` tendría una interpretación absoluta en vez de inferida.

## 3. De `pcov` a las incertezas de $\tau$ y $\beta$

`pcov` es una matriz $2\times2$ (una fila/columna por parámetro):

$$
\text{pcov} =
\begin{pmatrix}
\sigma_\tau^2 & \sigma_{\tau\beta} \\
\sigma_{\tau\beta} & \sigma_\beta^2
\end{pmatrix}
$$

- **Diagonal** = varianza de cada parámetro individual. La incerteza estándar (1$\sigma$) que guardamos en los `.txt` es:

$$
\sigma_\tau = \sqrt{\text{pcov}_{11}}, \qquad \sigma_\beta = \sqrt{\text{pcov}_{22}}
$$

  Esto es exactamente `np.sqrt(np.diag(pcov))`.

- **Fuera de la diagonal** ($\sigma_{\tau\beta}$) = covarianza entre $\tau$ y $\beta$. En una exponencial estirada es común que estén **correlacionados** (un $\tau$ más grande se puede "compensar" con un $\beta$ distinto y dar una curva parecida). El coeficiente de correlación es:

$$
\rho_{\tau\beta} = \frac{\sigma_{\tau\beta}}{\sigma_\tau \, \sigma_\beta} \in [-1,1]
$$

  Un $|\rho_{\tau\beta}|$ cercano a 1 significa que los dos parámetros individualmente son poco confiables aunque el ajuste se vea muy bueno — hay una "cresta" de $\chi^2$ casi plana en la dirección diagonal del espacio $(\tau,\beta)$.

## 4. Demostración numérica: reproduciendo `pcov` "a mano"

Abajo se genera un dataset sintético con la misma forma funcional, se ajusta con `curve_fit`, y se recalcula `pcov` manualmente con la fórmula $s^2 (J^TJ)^{-1}$ usando el Jacobiano numérico, para verificar que da lo mismo que lo que devuelve SciPy.

In [ ]:
import numpy as np
from scipy.optimize import curve_fit

rng = np.random.default_rng(0)

P0, Asymptote = 1, 0
def exponential_fit(t, tau, Beta):
    return P0 * np.exp(-(t / tau) ** Beta) + Asymptote

# Datos sinteticos: valores "verdaderos" tau=5, beta=0.7 + ruido
tau_true, beta_true = 5.0, 0.7
t = np.linspace(0.01, 20, 60)
y_true = exponential_fit(t, tau_true, beta_true)
noise_sigma = 0.02
y = y_true + rng.normal(0, noise_sigma, size=t.size)

bounds = ([1e-6, 1e-6], [t[-1]*1000, np.inf])
popt, pcov_scipy = curve_fit(exponential_fit, t, y, p0=[1, 1], bounds=bounds, maxfev=10000)
tau_fit, beta_fit = popt
print(f"tau ajustado = {tau_fit:.4f}, beta ajustado = {beta_fit:.4f}")
print("pcov (scipy):\n", pcov_scipy)

In [ ]:
# Reconstruccion manual: Jacobiano numerico en el optimo
def jacobian(f, x, theta, eps=1e-6):
    theta = np.asarray(theta, dtype=float)
    J = np.zeros((len(x), len(theta)))
    f0 = f(x, *theta)
    for j in range(len(theta)):
        dtheta = theta.copy()
        step = eps * max(1.0, abs(theta[j]))
        dtheta[j] += step
        f1 = f(x, *dtheta)
        J[:, j] = (f1 - f0) / step
    return J

J = jacobian(exponential_fit, t, popt)
residuals = y - exponential_fit(t, *popt)
N, p = len(t), len(popt)
chi2 = np.sum(residuals**2)
s2 = chi2 / (N - p)                 # varianza residual (grados de libertad = N - p)

pcov_manual = s2 * np.linalg.inv(J.T @ J)
print("pcov (manual, s^2 * (J^T J)^-1):\n", pcov_manual)

print("\nDiferencia relativa scipy vs manual:")
print(np.abs((pcov_manual - pcov_scipy) / pcov_scipy))

In [ ]:
# Incertezas 1-sigma y correlacion
sigma_tau, sigma_beta = np.sqrt(np.diag(pcov_scipy))
rho = pcov_scipy[0, 1] / (sigma_tau * sigma_beta)

print(f"tau  = {tau_fit:.4f} +/- {sigma_tau:.4f}  (valor verdadero: {tau_true})")
print(f"beta = {beta_fit:.4f} +/- {sigma_beta:.4f}  (valor verdadero: {beta_true})")
print(f"correlacion tau-beta: rho = {rho:.3f}")

## 5. Resumen práctico

| Cantidad | Qué es | Cómo se calcula |
|---|---|---|
| $J$ | Sensibilidad del modelo a cada parámetro | Derivadas parciales (numéricas) de `exponential_fit` |
| $J^TJ$ | Curvatura aproximada de $\chi^2$ en el óptimo | Producto matricial |
| $s^2$ | Varianza residual (ruido no explicado) | $\chi^2_{\min}/(N-p)$ |
| `pcov` | Matriz de covarianza de $(\tau,\beta)$ | $s^2\,(J^TJ)^{-1}$ — lo devuelve `curve_fit` directamente |
| $\sigma_\tau,\sigma_\beta$ | Incertezas 1$\sigma$ reportadas | `np.sqrt(np.diag(pcov))` |

### Limitaciones a tener en cuenta en tu caso concreto

1. **Aproximación local/Gaussiana**: asume que $\chi^2$ es aproximadamente cuadrático cerca del mínimo. Si el mínimo es muy asimétrico (frecuente en $\beta$, que está acotado a valores positivos), el error real puede no ser simétrico. Para verificarlo con más rigor: bootstrap sobre los residuos, o perfiles de $\chi^2$.
2. **No pasaste `sigma=`**: la incerteza está *inferida* de los residuos del propio ajuste, no de una incerteza experimental independiente.
3. **Ajustaste sobre datos suavizados** (`smooth5`): el promedio móvil reduce el ruido punto a punto, así que $\chi^2_{\min}$ (y por lo tanto $\sigma_\tau,\sigma_\beta$) van a salir **más chicos** de lo que serían con los datos crudos — es un piso optimista, no la incerteza real de la medición.